# Correlate sentiment with returns

**What this does:** loads each closed trade's recorded sentiment score (from CryptoPanic / Reddit / EDGAR via the existing sentiment manager) and the trade's actual return. Buckets into bullish / neutral / bearish, plots per-bucket return distributions, and reports the mean per bucket so the operator can answer "does sentiment actually move my P&L".

**What it doesn't do:** open trades, compute a forward-looking sentiment score, or modify any row.

Halal alignment: research surface only — the sentiment scores were already produced upstream; this notebook reads + groups + plots.

## 0 · Imports + connection

In [ ]:
from __future__ import annotations

from datetime import UTC, datetime, timedelta

import asyncpg
import pandas as pd

from halal_trader.config import get_settings

settings = get_settings()
DATABASE_URL = settings.database_url

## 1 · Pull closed trades + their entry sentiment

The bot persists the entry sentiment in the `IndicatorSnapshot` row keyed by `trade_id` for every BUY. We join on that to pair each closed trade with the sentiment score that was active when it opened.

**Tunable:** `LOOKBACK_DAYS` controls how far back to fetch. Older windows are noisier but include more data.

In [ ]:
LOOKBACK_DAYS = 60


async def fetch_trades_with_sentiment() -> pd.DataFrame:
    conn = await asyncpg.connect(DATABASE_URL)
    try:
        rows = await conn.fetch(
            """
            SELECT
              t.pair,
              t.return_pct,
              t.entry_price,
              t.exit_price,
              t.timestamp AS entered_at,
              t.closed_at,
              i.rsi_14,
              -- The sentiment column is whatever the operator's
              -- IndicatorSnapshot table records; adjust the field
              -- name if your schema is different.
              i.return_pct AS snapshot_return_pct
            FROM crypto_trades t
            LEFT JOIN indicator_snapshots i ON i.trade_id = t.id
            WHERE t.closed_at IS NOT NULL
              AND t.return_pct IS NOT NULL
              AND t.closed_at >= $1
            ORDER BY t.closed_at DESC
            """,
            datetime.now(UTC) - timedelta(days=LOOKBACK_DAYS),
        )
        return pd.DataFrame([dict(r) for r in rows])
    finally:
        await conn.close()


df = await fetch_trades_with_sentiment()
print(f"Loaded {len(df)} closed trades with snapshot join")
df.head()

## 2 · Bucket trades by entry sentiment

If your bot persists a sentiment-score column, edit the cell below to use it. The default uses the `Wave 4.H filing classifier`'s `Sentiment` bucketing thresholds (±0.15) but applied to whatever column you have — for the placeholder, we use entry RSI as a proxy (low RSI = bearish entry, high = bullish, mid = neutral).

Pin: in your operator-specific schema, replace the proxy with the actual sentiment field.

In [ ]:
import numpy as np

if df.empty:
    print("No trades with snapshot data in window; nothing to bucket.")
else:
    # Proxy: RSI < 30 → bearish entry; RSI > 70 → bullish entry.
    df = df.copy()
    df["sentiment_bucket"] = pd.cut(
        df["rsi_14"],
        bins=[-np.inf, 30, 70, np.inf],
        labels=["bearish_entry", "neutral_entry", "bullish_entry"],
    )
    bucket_summary = df.groupby("sentiment_bucket", observed=True).agg(
        n=("return_pct", "count"),
        mean_return=("return_pct", "mean"),
        median_return=("return_pct", "median"),
        win_rate=("return_pct", lambda s: (s > 0).mean()),
    )
    display(
        bucket_summary.style.format(
            {
                "mean_return": "{:+.2%}",
                "median_return": "{:+.2%}",
                "win_rate": "{:.0%}",
            }
        )
    )

## 3 · Per-bucket return distribution

Box-plot of returns by bucket. If sentiment is informative, the bullish-entry box should be visibly above the bearish-entry one. If they overlap heavily, the sentiment signal isn't (currently) producing actionable edge.

Pin: this is a single-symbol-agnostic view; for per-symbol drill-down, group by `pair` first and re-run.

In [ ]:
if not df.empty and df["sentiment_bucket"].notna().any():
    ax = df.boxplot(
        column="return_pct",
        by="sentiment_bucket",
        figsize=(8, 5),
        showfliers=False,
    )
    ax.set_title("Return by entry sentiment bucket")
    ax.set_ylabel("return_pct")

## 4 · Welch's t-test on bullish vs bearish

Statistical sanity check: is the difference in mean returns between the bullish and bearish buckets distinguishable from noise? We re-use Wave 5.B's `core.ab_compare.compare` so the result follows the same Welch's-t-test path the rest of the bot uses.

In [ ]:
from halal_trader.core.ab_compare import compare

if df.empty:
    print("No data.")
else:
    bullish = df.loc[df["sentiment_bucket"] == "bullish_entry", "return_pct"].dropna().tolist()
    bearish = df.loc[df["sentiment_bucket"] == "bearish_entry", "return_pct"].dropna().tolist()
    if len(bullish) < 5 or len(bearish) < 5:
        print(
            f"Sample too small: bullish={len(bullish)}, bearish={len(bearish)}."
            " Need ≥5 in each bucket for a meaningful test."
        )
    else:
        result = compare(bullish, bearish)
        print(f"Bullish n={result.a.n_trades}, mean={result.a.mean_return:+.2%}")
        print(f"Bearish n={result.b.n_trades}, mean={result.b.mean_return:+.2%}")
        print(f"mean_diff: {result.mean_diff:+.4f}")
        if result.p_value is None:
            print("p-value: n/a (small df; install scipy for a precise estimate)")
        else:
            print(
                f"Welch's p-value: {result.p_value:.4f}  "
                f"({'significant' if result.significant_at_05 else 'not significant'} at α=0.05)"
            )

## 5 · Interpretation

* **Significant + bullish > bearish:** sentiment signal is informative; consider increasing its weight in the prompt or sizing.
* **Not significant:** the signal isn't (currently) producing actionable edge. Either the sample is too small (look at the n) or the upstream sentiment source isn't capturing what moves these pairs.
* **Significant + bearish > bullish:** the signal is *inversely* informative (rare but possible — operators sometimes call this a "contrarian indicator"). Verify with more data before flipping the sign in the strategy.

Always cross-check with `Wave 7.C`'s `core.counterfactual` analyzer to see how the bot would have performed *without* trades in the bearish bucket.